# Scikit SGD: Minimal Training Pipeline

This notebook is the first concrete implementation of the Bovi Core training contracts. It trains a small `SGDRegressor` entirely on CPU and demonstrates configuration, data loading, transforms, training context, epoch results, local checkpoints, evaluation, and resume.

The example deliberately keeps orchestration and logging outside the trainer.

## Section 1: Configuration

In [ ]:
from pathlib import Path

from bovi_core.config import Config
from bovi_core.ml import ModelProviderRegistry

Config.reset()
config = Config(experiment_name="scikit_sgd", project_name="scikit-sgd")

print(f"Environment: {config.environment}")
print(f"Project root: {config.project.project_root}")
print(f"Experiment: {config.experiment.experiment_name}")
print(f"Available providers: {ModelProviderRegistry.list_available()}")

In [ ]:
from scikit_sgd import (
    ScikitSGDEvaluationConfig,
    ScikitSGDModelConfig,
    ScikitSGDTrainingConfig,
)

model_config = ScikitSGDModelConfig.from_config(config)
training_config = ScikitSGDTrainingConfig.from_config(config)
evaluation_config = ScikitSGDEvaluationConfig.from_config(config)

print(f"Model config: {model_config}")
print(f"Training config: {training_config}")
print(f"Evaluation config: {evaluation_config}")

## Section 2: Config-driven data pipelines

In [ ]:
from scikit_sgd import create_dataloader

dataloaders = {
    split: create_dataloader(config, model_config, split) for split in ("train", "validation")
}

for split, dataloader in dataloaders.items():
    print(
        f"{split}: {dataloader.num_samples} samples, "
        f"{dataloader.num_batches} batches, batch_size={dataloader.batch_size}"
    )

In [ ]:
batch = next(iter(dataloaders["train"]))

print("Transformed and collated batch:")
for name, values in batch["features"].items():
    print(
        f"  features.{name}: shape={values.shape}, range=({values.min():.3f}, {values.max():.3f})"
    )
print(f"  labels: shape={batch['labels'].shape}")
print(f"  metadata records: {len(batch['metadata'])}")

## Section 3: Fresh model and training context

In [ ]:
provider = ModelProviderRegistry.create("scikit_sgd")
model = provider.create(model_config)

print(f"Runtime wrapper: {type(model).__name__}")
print(f"Native estimator: {type(model.native_model).__name__}")
print(f"Fitted before training: {hasattr(model.native_model, 'coef_')}")

In [ ]:
from tempfile import gettempdir
from uuid import uuid4

from bovi_core.ml import TrainingContext

run_id = uuid4()
output_dir = Path(gettempdir()) / "bovi-scikit-sgd" / str(run_id)
context = TrainingContext(
    run_id=run_id,
    reason="notebook demonstration",
    output_dir=output_dir,
)

print(f"Run ID: {context.run_id}")
print(f"Output directory: {context.output_dir}")

## Section 4: Train and inspect the result

In [ ]:
from scikit_sgd import ScikitSGDTrainer

trainer = ScikitSGDTrainer(
    model=model,
    dataloaders=dataloaders,
    config=training_config,
    context=context,
)
training_result = trainer.train()

print(f"Status: {training_result.status}")
print(f"Stop reason: {training_result.stop_reason}")
print(f"Epochs completed: {len(training_result.epochs)}")
print(f"Best epoch: {training_result.best_epoch}")
print(f"Last checkpoint: {training_result.last_checkpoint.uri}")
print(f"Best checkpoint: {training_result.best_checkpoint.uri}")

In [ ]:
import json

from bovi_core.ml.trainers import LocalTrainingResultLogger, TrainingContext, TrainingResult

run_metadata = {
    "experiment": "scikit_sgd",
    "splits": {
        split: {
            "records": loader.num_samples,
            "batches": len(loader),
            "batch_size": loader.batch_size,
        }
        for split, loader in dataloaders.items()
    },
}
logger = LocalTrainingResultLogger(
    metadata=run_metadata,
    config_snapshot={
        "model": model_config.model_dump(mode="json"),
        "training": training_config.model_dump(mode="json"),
    },
)
log_outcome = await logger.log(context, training_result)
print(log_outcome.model_dump_json(indent=2))
assert log_outcome.destinations[0].status == "success", log_outcome
manifest_path = context.output_dir / "training-results" / f"{context.run_id}.json"
saved_manifest = json.loads(manifest_path.read_text())
saved_context = TrainingContext.model_validate(saved_manifest["context"])
saved_result = TrainingResult.model_validate(saved_manifest["result"])
assert saved_context == context
assert saved_result == training_result
assert saved_manifest["config_snapshot"]["training"] == training_config.model_dump(mode="json")
print("Stored manifest:", manifest_path)

In [ ]:
import matplotlib.pyplot as plt

epochs = [result.epoch for result in training_result.epochs]
train_mse = [result.metrics["train_mse"] for result in training_result.epochs]
validation_mse = [result.metrics["validation_mse"] for result in training_result.epochs]

plt.figure(figsize=(8, 4))
plt.plot(epochs, train_mse, label="train MSE")
plt.plot(epochs, validation_mse, label="validation MSE")
plt.axvline(training_result.best_epoch, color="black", linestyle="--", label="best epoch")
plt.xlabel("Epoch within this attempt")
plt.ylabel("MSE")
plt.legend()
plt.tight_layout()
plt.show()

print(f"First validation MSE: {validation_mse[0]:.3f}")
print(f"Best validation MSE: {min(validation_mse):.3f}")
print(f"Final validation MSE: {validation_mse[-1]:.3f}")

## Section 5: Evaluate independently

In [ ]:
from bovi_core.ml import EvaluationContext
from scikit_sgd import ScikitSGDEvaluator

evaluation_context = EvaluationContext(
    evaluation_id=uuid4(),
    split="validation",
    model_version="notebook-last",
    training_run_id=run_id,
    output_dir=output_dir / "evaluation",
)
evaluation_result = ScikitSGDEvaluator(model, evaluation_config).evaluate(
    dataloaders["validation"],
    evaluation_context,
)

print(f"Status: {evaluation_result.status}")
print(f"Examples: {evaluation_result.num_examples}")
for name, value in evaluation_result.metrics.items():
    print(f"  {name}: {value:.4f}")

## Section 6: Restore and resume a new attempt

In [ ]:
from bovi_core.ml.models.checkpoints import LocalCheckpointResolver

resolved_checkpoint = LocalCheckpointResolver().resolve(saved_result.last_checkpoint)
resumed_model = provider.restore_checkpoint(model_config, resolved_checkpoint)

resume_run_id = uuid4()
resume_context = TrainingContext(
    run_id=resume_run_id,
    resumed_from_run_id=run_id,
    reason="resume demonstration",
    output_dir=Path(gettempdir()) / "bovi-scikit-sgd" / str(resume_run_id),
)
resume_config = training_config.model_copy(update={"epochs": 2, "early_stopping_patience": None})
resume_result = ScikitSGDTrainer(
    model=resumed_model,
    dataloaders=dataloaders,
    config=resume_config,
    context=resume_context,
).train()

print(f"Resumed from: {resume_context.resumed_from_run_id}")
print(f"New run ID: {resume_result.run_id}")
print(f"Epochs in new attempt: {[epoch.epoch for epoch in resume_result.epochs]}")
print(f"Status: {resume_result.status}")

In [ ]:
resume_logger = LocalTrainingResultLogger(
    metadata=run_metadata,
    config_snapshot={
        "model": model_config.model_dump(mode="json"),
        "training": resume_config.model_dump(mode="json"),
    },
)
resume_log_outcome = await resume_logger.log(resume_context, resume_result)
print(resume_log_outcome.model_dump_json(indent=2))
assert resume_log_outcome.destinations[0].status == "success", resume_log_outcome
resume_manifest_path = (
    resume_context.output_dir / "training-results" / f"{resume_context.run_id}.json"
)
resume_manifest = json.loads(resume_manifest_path.read_text())
assert resume_manifest["context"]["resumed_from_run_id"] == str(context.run_id)
assert resume_manifest["config_snapshot"]["training"] == resume_config.model_dump(mode="json")
assert TrainingResult.model_validate(resume_manifest["result"]) == resume_result
print("Stored resume manifest:", resume_manifest_path)

## Summary

The concrete pipeline is:

`YAML -> typed configs -> JSON source -> transforms -> dataset -> dataloaders -> provider -> model -> trainer -> TrainingResult -> evaluator`

Important boundaries demonstrated here:

- The model config defines the estimator's stable input contract.
- The training config defines one attempt's runtime behavior.
- The context carries identity, output location, reason, and an optional deadline.
- Every attempt numbers its epochs from one, including a resumed attempt.
- The result contains metrics and checkpoint references, not heavyweight model copies.
- Evaluation remains separate from training.
- An orchestrator can replace this local flow with a federated flow without changing the trainer contract.